In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split


In [2]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
train_df.head()


Total Users : 1760
Total Items : 2881


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [3]:
# Data loaded and remapped above.
print(f"Train rows : {len(train_df)}")
print(f"Val rows   : {len(val_df)}")
print(f"Test rows  : {len(test_df)}")
print(f"Rating col : {train_df.columns[2]}  "
      f"range [{train_df.iloc[:,2].min()}, {train_df.iloc[:,2].max()}]")


Train rows : 62568
Val rows   : 15643
Test rows  : 19553
Rating col : bought  range [1, 24]


In [4]:
# ── Dataset & DataLoader (self-contained, no src dependency) ─────────────────
class RatingDataset(Dataset):
    """Wraps a DataFrame with columns [user_id, item_id, rating].
    Column positions 0,1,2 are used so any column names work.
    """
    def __init__(self, df):
        arr = df.iloc[:, :3].values
        self.users   = torch.tensor(arr[:, 0], dtype=torch.long)
        self.items   = torch.tensor(arr[:, 1], dtype=torch.long)
        self.ratings = torch.tensor(arr[:, 2], dtype=torch.float32)

    def __len__(self):  return len(self.ratings)

    def __getitem__(self, idx):
        return (torch.stack([self.users[idx], self.items[idx]], dim=0),
                self.ratings[idx])


def make_loader(df, batch_size, shuffle=True):
    return DataLoader(RatingDataset(df), batch_size=batch_size, shuffle=shuffle)


In [5]:
# ── Matrix Factorisation model ────────────────────────────────────────────────
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super().__init__()
        self.user_emb  = nn.Embedding(num_users, embedding_dim)
        self.item_emb  = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

        nn.init.normal_(self.user_emb.weight,  std=0.1)
        nn.init.normal_(self.item_emb.weight,  std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids):
        u   = self.user_emb(user_ids)
        v   = self.item_emb(item_ids)
        dot = (u * v).sum(1, keepdim=True)
        return (dot + self.user_bias(user_ids)
                    + self.item_bias(item_ids)
                    + self.global_bias).squeeze()


In [6]:
# ── Training & validation loops ──────────────────────────────────────────────
def train_loop(model, loader, optimizer, device='cpu'):
    loss_fn = nn.MSELoss(reduction='mean')
    model.train()
    total_loss, total_n = 0.0, 0
    for inputs, target in tqdm(loader, leave=False):
        inputs, target = inputs.to(device), target.to(device)
        optimizer.zero_grad()
        pred = model(inputs[:, 0], inputs[:, 1])
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(target)
        total_n    += len(target)
    return np.sqrt(total_loss / total_n)   # train RMSE


def validate_loop(model, loader, device='cpu'):
    loss_fn = nn.MSELoss(reduction='sum')
    model.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for inputs, target in loader:
            inputs, target = inputs.to(device), target.to(device)
            pred        = model(inputs[:, 0], inputs[:, 1])
            total_loss += loss_fn(pred, target).item()
            total_n    += len(target)
    # FIX: return RMSE directly (original double-sqrt bug removed)
    return np.sqrt(total_loss / total_n)   # val RMSE


In [7]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
batch_size   = 64
n_factors    = 30
lr           = 1e-2
weight_decay = 2e-3
momentum     = 0.1
n_epochs     = 50
patience     = 5
device       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = float('inf')
        self.counter   = 0
    def step(self, val_loss):
        if val_loss < self.best - self.min_delta:
            self.best    = val_loss
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


device: cpu


## Parameter Tuning — Grid Search

Grid over `n_factors`, `lr`, and `weight_decay`.
Each combo trains for up to 20 epochs with patience-3 on val RMSE.
Writes `n_factors`, `lr`, `weight_decay` for the full training run.


In [ ]:
from itertools import product as iproduct

N_FACTORS_GRID   = [16, 32, 64]
LR_GRID          = [1e-3, 1e-2, 5e-2]
WEIGHT_DECAY_GRID= [1e-4, 1e-3, 1e-2]
TUNE_EPOCHS      = 20
TUNE_PAT         = 3
TUNE_BATCH       = 256   # larger batch for speed during tuning

param_grid = list(iproduct(N_FACTORS_GRID, LR_GRID, WEIGHT_DECAY_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'factors':>8}  {'lr':>8}  {'wd':>8}  {'best val RMSE':>14}")
print("-" * 48)

# Fixed during tuning
_momentum   = 0.1
_tune_dl    = make_loader(train_df, batch_size=TUNE_BATCH, shuffle=True)
_tune_val   = make_loader(val_df,   batch_size=512,        shuffle=False)

grid_results = []

for _k, (_nf, _lr, _wd) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    _model = MatrixFactorization(n_users, n_items, embedding_dim=_nf).to(device)
    _opt   = SGD(_model.parameters(), lr=_lr, weight_decay=_wd, momentum=_momentum)
    _es    = EarlyStopper(patience=TUNE_PAT)
    _best_val = float('inf')

    for _ep in range(TUNE_EPOCHS):
        train_loop(_model, _tune_dl, _opt, device)
        _v = validate_loop(_model, _tune_val, device)
        if _v < _best_val:
            _best_val = _v
        if _es.step(_v):
            break

    grid_results.append((_best_val, _nf, _lr, _wd))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_nf:>8d}  {_lr:>8.4f}  {_wd:>8.5f}  {_best_val:>14.4f}{_marker}")

_best = min(grid_results, key=lambda x: x[0])
best_val_rmse_tune, best_nf, best_lr, best_wd = _best
print(f"\nBest val RMSE  : {best_val_rmse_tune:.4f}")
print(f"  n_factors      = {best_nf}")
print(f"  lr             = {best_lr}")
print(f"  weight_decay   = {best_wd}")
n_factors    = best_nf
lr           = best_lr
weight_decay = best_wd
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combinations x up to 20 epochs
  #   factors        lr        wd   best val RMSE
------------------------------------------------


  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

...

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

  0%|          | 0/245 [00:00<?, ?it/s]

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
# Centralized: pool all users' data into a single loader
train_dl = make_loader(train_df, batch_size=batch_size, shuffle=True)
val_dl   = make_loader(val_df,   batch_size=256,        shuffle=False)
test_dl  = make_loader(test_df,  batch_size=256,        shuffle=False)

print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)} | Test batches: {len(test_dl)}")


In [ ]:
# ── Model, optimiser, early stopping ─────────────────────────────────────────
# Fall back to defaults if grid search was skipped
try: n_factors
except NameError: n_factors = 30
try: lr
except NameError: lr = 1e-2
try: weight_decay
except NameError: weight_decay = 2e-3

model     = MatrixFactorization(n_users, n_items, embedding_dim=n_factors).to(device)
optimizer = SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=momentum)
early_stopper = EarlyStopper(patience=patience)


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
train_losses, val_losses = [], []

for epoch in range(1, n_epochs + 1):
    tr_rmse  = train_loop(model, train_dl, optimizer, device)
    val_rmse = validate_loop(model, val_dl, device)
    train_losses.append(tr_rmse)
    # FIX: val_rmse is already RMSE — do NOT take sqrt again
    val_losses.append(val_rmse)
    print(f"Epoch {epoch:3d} | Train RMSE: {tr_rmse:.4f} | Val RMSE: {val_rmse:.4f}")
    if early_stopper.step(val_rmse):
        print("Early stopping.")
        break


In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_rmse = validate_loop(model, test_dl, device)
print(f"Test RMSE: {test_rmse:.4f}")


In [ ]:
# ── Save per-epoch log ───────────────────────────────────────────────────────
import pandas as pd
log_df = pd.DataFrame({
    "epoch":      list(range(1, len(val_losses) + 1)),
    "train_rmse": train_losses,
    "val_rmse":   val_losses,
})
log_df.to_csv("cmf_hm.csv", index=False)
print(f"Saved cmf_hm.csv  ({len(log_df)} epochs)")
log_df
